# Long Short-Term Memory (LSTM)

Neste notebook, exploraremos a arquitetura de Redes Neurais Recorrentes do tipo **Long Short-Term Memory (LSTM)**. A LSTM foi projetada especificamente para solucionar o problema do desvanecimento e explosão do gradiente (*vanishing and exploding gradients*) em sequências longas, permitindo que a rede retenha informações por muitos passos temporais.

Construiremos um pipeline unificado abordando duas aplicações fundamentais em Processamento de Linguagem Natural (PLN):
1. **Classificação de Texto**: Detecção de Spam em mensagens SMS utilizando o dataset *sms_spam*.
2. **Geração Autoregressiva**: Criação de novos nomes de Pokémon baseando-se em modelagem a nível de caractere.

## Matemática da Célula LSTM

Diferente de uma célula RNN simples, a célula LSTM possui uma estrutura interna mais complexa chamada **estado da célula** ($C_t$), que age como uma "esteira de informações", e três **portões** (*gates*) que controlam o fluxo de leitura, escrita e exclusão de informações.

As equações que regem uma célula LSTM em cada passo de tempo $t$ são dadas por:

1. **Portão de Esquecimento (*Forget Gate*)**:
   $$f_t = \sigma(W_f \cdot [h_{t-1}, x_t] + b_f)$$
   Determina qual fração da informação anterior do estado da célula ($C_{t-1}$) deve ser descartada.

2. **Portão de Entrada (*Input Gate*)**:
   $$i_t = \sigma(W_i \cdot [h_{t-1}, x_t] + b_i)$$
   Controla qual nova informação será armazenada no estado da célula.

3. **Candidato a Novo Estado de Célula**:
   $$\tilde{C}_t = \tanh(W_c \cdot [h_{t-1}, x_t] + b_c)$$
   Gera novos valores candidatos que podem ser adicionados ao estado da célula.

4. **Atualização do Estado da Célula (*Cell State*)**:
   $$C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$$
   Atualiza a memória de longo prazo combinando o estado anterior atenuado e os novos candidatos.

5. **Portão de Saída (*Output Gate*)**:
   $$o_t = \sigma(W_o \cdot [h_{t-1}, x_t] + b_o)$$
   Decide qual parte do estado da célula atualizado será revelada na saída.

6. **Estado Oculto Atual (*Hidden State*)**:
   $$h_t = o_t \odot \tanh(C_t)$$
   Calcula a memória de curto prazo e a saída imediata da célula, passando o estado filtrado pela ativação $\tanh$.

Onde:
* $x_t$ é o vetor de entrada no passo $t$.
* $h_{t-1}$ é o estado oculto do passo anterior.
* $W$ e $b$ são pesos e vieses a serem aprendidos.
* $\sigma$ representa a função de ativação sigmoide (valores entre 0 e 1).
* $\odot$ representa o produto de Hadamard (multiplicação elemento a elemento).

In [ ]:
import re
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from datasets import load_dataset
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

In [ ]:
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Dispositivo selecionado: {device}")

## Classe Tokenizadora

Utilizaremos a **mesma classe tokenizadora** para ambas as tarefas deste notebook. Esta classe recebe uma função de separação (`split_fn`), monta o vocabulário a partir de uma coleção de textos e mantém os dicionários internos de conversão de tokens para índices e vice-versa.

In [ ]:
class Tokenizer:
    def __init__(self, split_fn, min_freq=1, special_tokens=None):
        self.split_fn = split_fn
        self.min_freq = min_freq
        self.special_tokens = special_tokens if special_tokens is not None else ["<pad>", "<unk>"]
        self.vocab = {}
        self.idx_to_token = {}

    def build_vocab(self, texts):
        counter = Counter()
        for text in texts:
            counter.update(self.split_fn(text))
        
        # Mapeia tokens especiais no início do vocabulário
        self.vocab = {tok: idx for idx, tok in enumerate(self.special_tokens)}
        
        for tok, freq in counter.items():
            if freq >= self.min_freq and tok not in self.vocab:
                self.vocab[tok] = len(self.vocab)
                
        self.idx_to_token = {idx: tok for tok, idx in self.vocab.items()}

    def encode(self, text):
        tokens = self.split_fn(text)
        return [self.vocab.get(tok, self.vocab.get("<unk>", 0)) for tok in tokens]

    def decode(self, indices):
        return [self.idx_to_token.get(idx, "<unk>") for idx in indices]

In [ ]:
test_texts = [
    "Eu gosto de PyTorch",
    "PyTorch e Deep Learning sao legais",
    "Instituto Metrópole Digital"
]

def dummy_split_fn(text):
    return text.lower().split()

dummy_tokenizer = Tokenizer(split_fn=dummy_split_fn, min_freq=1, special_tokens=["<pad>", "<unk>"])
dummy_tokenizer.build_vocab(test_texts)

In [ ]:
# Exibe o vocabulário gerado
print("Vocabulário:")
print(dummy_tokenizer.vocab)

In [ ]:
# Testando a Codificação (Text -> Indices)
frase = "Eu gosto de Machine Learning"
indices = dummy_tokenizer.encode(frase)
print(f"Frase original: '{frase}'")
print(f"Índices gerados: {indices}")

In [ ]:
# Testando a Decodificação (Indices -> Tokens)
tokens_decodificados = dummy_tokenizer.decode(indices)
print(f"Tokens decodificados: {tokens_decodificados}")

## Classificação de SMS com LSTM

Nesta subseção, implementaremos um classificador de mensagens SMS usando uma rede LSTM. O modelo decidirá se uma mensagem é **Ham** (legítima) ou **Spam** (indesejada).

### Preparação dos Dados
Carregamos o dataset `sms_spam` e dividimos as amostras em 80% para treino e 20% para teste. Instanciamos o `Tokenizer` a nível de palavras e estruturamos os `DataLoader` de treino e validação aplicando a colagem com preenchimento (`collate_batch`).

In [ ]:
raw_dataset = load_dataset("sms_spam")["train"]
texts = raw_dataset["sms"]
labels = raw_dataset["label"]

In [ ]:
np.random.seed(42)
indices = np.random.permutation(len(texts))
split = int(0.8 * len(texts))

train_texts = [texts[i] for i in indices[:split]]
train_labels = [labels[i] for i in indices[:split]]
test_texts = [texts[i] for i in indices[split:]]
test_labels = [labels[i] for i in indices[split:]]

print(f"Amostras de treino: {len(train_texts)}")
print(f"Amostras de teste: {len(test_texts)}")

In [ ]:
print("Exemplo de SMS legítimo (Ham):")
print(next(t for t, l in zip(train_texts, train_labels) if l == 0).strip())

print("\nExemplo de SMS indesejado (Spam):")
print(next(t for t, l in zip(train_texts, train_labels) if l == 1).strip())

In [ ]:
token_pattern = re.compile(r"\b\w+\b", flags=re.UNICODE)

def word_split_fn(text):
    return token_pattern.findall(text.lower())

sms_tokenizer = Tokenizer(split_fn=word_split_fn, min_freq=2)
sms_tokenizer.build_vocab(train_texts)

print(f"Tamanho do vocabulário: {len(sms_tokenizer.vocab)}")

In [ ]:
class SMSDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        indices = self.tokenizer.encode(self.texts[idx])
        if not indices:
            indices = [self.tokenizer.vocab["<unk>"]]
        return torch.tensor(indices, dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.long)

train_dataset = SMSDataset(train_texts, train_labels, sms_tokenizer)
test_dataset = SMSDataset(test_texts, test_labels, sms_tokenizer)

In [ ]:
def collate_batch(batch):
    text_list, label_list = [], []
    for _text_indices, _label in batch:
        text_list.append(_text_indices)
        label_list.append(_label)
    
    text_padded = pad_sequence(text_list, batch_first=True, padding_value=0)
    labels = torch.tensor(label_list, dtype=torch.long)
    return text_padded.to(device), labels.to(device)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_batch)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_batch)

### Arquitetura do Modelo
Implementamos o modelo `SMSClassifier` contendo uma camada `nn.Embedding`, uma camada `nn.LSTM` básica de 1 camada e a camada linear final. Instanciamos a rede, a perda e o otimizador.

In [ ]:
class SMSClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        embedded = self.embedding(x)
        _, (hidden, _) = self.lstm(embedded)
        # hidden[-1] representa o estado oculto final
        return self.fc(hidden[-1])

In [ ]:
model_clf = SMSClassifier(len(sms_tokenizer.vocab), embed_dim=64, hidden_dim=128).to(device)

In [ ]:
criterion_clf = nn.CrossEntropyLoss()
optimizer_clf = optim.Adam(model_clf.parameters(), lr=0.002)

### Treinamento e Avaliação
Treinamos a rede LSTM ao longo de 8 épocas. Após o ajuste dos parâmetros, avaliamos o erro e a acurácia, plotamos a curva de aprendizado, geramos a matriz de confusão e realizamos predições de teste.

In [ ]:
epochs = 8
train_losses, val_losses = [], []

for epoch in range(epochs):
    model_clf.train()
    train_loss, correct, total = 0.0, 0, 0
    for x, y in train_loader:
        optimizer_clf.zero_grad()
        preds = model_clf(x)
        loss = criterion_clf(preds, y)
        loss.backward()
        optimizer_clf.step()
        
        train_loss += loss.item()
        correct += (preds.argmax(dim=1) == y).sum().item()
        total += y.size(0)
    
    # Validação
    model_clf.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for x, y in test_loader:
            preds = model_clf(x)
            loss = criterion_clf(preds, y)
            val_loss += loss.item()
            val_correct += (preds.argmax(dim=1) == y).sum().item()
            val_total += y.size(0)
            
    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(test_loader)
    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)
    
    print(f"Época {epoch+1:02d} | Loss Treino: {avg_train_loss:.4f} | Acc Treino: {correct/total:.2%} | Loss Val: {avg_val_loss:.4f} | Acc Val: {val_correct/val_total:.2%}")

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(train_losses, label="Treino", color="blue", marker="o")
plt.plot(val_losses, label="Validação", color="orange", marker="s")
plt.title("Curva de Perda")
plt.xlabel("Época")
plt.ylabel("Perda")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
model_clf.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for x, y in test_loader:
        preds = model_clf(x).argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(y.cpu().numpy())

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Ham", "Spam"], yticklabels=["Ham", "Spam"])
plt.xlabel("Predito")
plt.ylabel("Real")
plt.title("Matriz de Confusão")
plt.show()

In [ ]:
def predict_sms(text):
    model_clf.eval()
    indices = sms_tokenizer.encode(text)
    if not indices:
        indices = [sms_tokenizer.vocab["<unk>"]]
    tensor = torch.tensor(indices, dtype=torch.long).unsqueeze(0).to(device)
    
    with torch.no_grad():
        logits = model_clf(tensor)
        probs = torch.softmax(logits, dim=1)
        pred_idx = probs.argmax().item()
        
    classes = ["Ham (Não Spam)", "Spam"]
    return classes[pred_idx], probs[0][pred_idx].item()

In [ ]:
mensagens = [
    "Hey! Are we still meeting for lunch today at 1 pm?",
    "URGENT! Your mobile number has been selected for a £2000 prize! Call 09061221191 now!"
]

for msg in mensagens:
    classe, conf = predict_sms(msg)
    print(f"Mensagem: '{msg}'\nPredição: {classe} ({conf:.2%})\n")

## Geração Autoregressiva de Texto

Nesta subseção, utilizaremos a rede LSTM para modelagem em nível de caractere. O modelo aprenderá as dependências temporais entre caracteres consecutivos para gerar nomes inéditos baseados em fonemas de Pokémon.

### Preparação dos Dados
Definimos a lista de nomes dos Pokémons, instanciamos o `Tokenizer` a nível de caracteres (com as tags `<sos>` e `<eos>`), e criamos o dataset e dataloader com preenchimento em lote para modelagem autoregressiva.

In [ ]:
pokemons = [
    "bulbasaur", "ivysaur", "venusaur", "charmander", "charmeleon", "charizard", "squirtle", "wartortle", 
    "blastoise", "caterpie", "metapod", "butterfree", "weedle", "kakuna", "beedrill", "pidgey", "pidgeotto", 
    "pidgeot", "rattata", "raticate", "spearow", "fearow", "ekans", "arbok", "pikachu", "raichu", "sandshrew", 
    "sandslash", "nidoran", "nidorina", "nidoqueen", "nidorino", "nidoking", "clefairy", "clefable", "vulpix", 
    "ninetales", "jigglypuff", "wigglytuff", "zubat", "golbat", "oddish", "gloom", "vileplume", "paras", 
    "parasect", "venonat", "venomoth", "diglett", "dugtrio", "meowth", "persian", "psyduck", "golduck", 
    "mankey", "primeape", "growlithe", "arcanine", "poliwag", "poliwhirl", "poliwrath", "abra", "kadabra", 
    "alakazam", "machop", "machoke", "machamp", "bellsprout", "weepinbell", "victreebel", "tentacool", 
    "tentacruel", "geodude", "graveler", "golem", "ponyta", "rapidash", "slowpoke", "slowbro", "magnemite", 
    "magneton", "farfetchd", "doduo", "dodrio", "seel", "dewgong", "grimer", "muk", "shellder", "cloyster", 
    "gastly", "haunter", "gengar", "onix", "drowzee", "hypno", "krabby", "kingler", "voltorb", "electrode", 
    "exeggcute", "exeggutor", "cubone", "marowak", "hitmonlee", "hitmonchan", "lickitung", "koffing", "weezing", 
    "rhyhorn", "rhydon", "chansey", "tangela", "kangaskhan", "horsea", "seadra", "goldeen", "seaking", "staryu", 
    "starmie", "mrmime", "scyther", "jynx", "electabuzz", "magmar", "pinsir", "tauros", "magikarp", "gyarados", 
    "lapras", "ditto", "eevee", "vaporeon", "jolteon", "flareon", "porygon", "omanyte", "omastar", "kabuto", 
    "kabutops", "aerodactyl", "snorlax", "articuno", "zapdos", "moltres", "dratini", "dragonair", "dragonite", 
    "mewtwo", "mew"
]
print(f"Total de Pokémons: {len(pokemons)}")

In [ ]:
# import gdown

# filepath = "data/pokemons.txt"

# gdown.download(
#     "https://drive.google.com/uc?id=1XSCItDv6aJ9myb3d1p6iHPLivOAo42k8",
#     filepath, quiet=False
# )

# with open(filepath) as f:
#     pokemons = f.read().split("\n")

# print(f"Total de Pokémons: {len(pokemons)}")

In [ ]:
def char_split_fn(text):
    return list(text.lower())

pokemon_tokenizer = Tokenizer(split_fn=char_split_fn, min_freq=1, special_tokens=["<pad>", "<unk>", "<sos>", "<eos>"])
pokemon_tokenizer.build_vocab(pokemons)

print(f"Tamanho do vocabulário de caracteres: {len(pokemon_tokenizer.vocab)}")

In [ ]:
class PokemonDataset(Dataset):
    def __init__(self, names, tokenizer):
        self.names = names
        self.tokenizer = tokenizer
        self.sos_idx = tokenizer.vocab["<sos>"]
        self.eos_idx = tokenizer.vocab["<eos>"]

    def __len__(self):
        return len(self.names)

    def __getitem__(self, idx):
        encoded = self.tokenizer.encode(self.names[idx])
        x = [self.sos_idx] + encoded
        y = encoded + [self.eos_idx]
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)

pokemon_dataset = PokemonDataset(pokemons, pokemon_tokenizer)

In [ ]:
def collate_pokemon(batch):
    x_list, y_list = zip(*batch)
    x_padded = pad_sequence(x_list, batch_first=True, padding_value=0)
    y_padded = pad_sequence(y_list, batch_first=True, padding_value=0)
    return x_padded.to(device), y_padded.to(device)

pokemon_loader = DataLoader(pokemon_dataset, batch_size=16, shuffle=True, collate_fn=collate_pokemon)

In [ ]:
pokemon_dataset[0]

out, state = model_gen(pokemon_dataset[0][0].to(device))
out.shape

### Arquitetura do Modelo
Implementamos o modelo gerador `LSTMGenerator` usando uma camada `nn.Embedding`, uma camada `nn.LSTM` básica de 1 camada e a camada linear de saída. Instanciamos o gerador, configurando a perda para ignorar os paddings (`ignore_index=0`) e o otimizador.

In [ ]:
class LSTMGenerator(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, state=None):
        embedded = self.embedding(x)
        out, state = self.lstm(embedded, state)
        return self.fc(out), state

In [ ]:
gen_vocab_size = len(pokemon_tokenizer.vocab)
model_gen = LSTMGenerator(gen_vocab_size, embed_dim=32, hidden_dim=128).to(device)

In [ ]:
criterion_gen = nn.CrossEntropyLoss(ignore_index=0)
optimizer_gen = optim.Adam(model_gen.parameters(), lr=0.005)

### Treinamento e Amostragem
Treinamos a rede geradora ao longo de 30 épocas. Em seguida, implementamos a amostragem probabilística escalada pelo parâmetro de Temperatura e geramos novos nomes de Pokémon.

In [ ]:
gen_epochs = 30
model_gen.train()

for epoch in range(gen_epochs):
    epoch_loss = 0.0
    for x, y in pokemon_loader:
        optimizer_gen.zero_grad()
        logits, _ = model_gen(x)
        
        # Redimensiona logits e targets para calcular erro multiclasse
        loss = criterion_gen(logits.view(-1, gen_vocab_size), y.view(-1))
        loss.backward()
        optimizer_gen.step()
        
        epoch_loss += loss.item()
        
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Época {epoch+1:02d}/{gen_epochs} | Loss: {epoch_loss/len(pokemon_loader):.4f}")

In [ ]:
def generate_pokemon_name():
    model_gen.eval()
    sos_idx = pokemon_tokenizer.vocab["<sos>"]
    eos_idx = pokemon_tokenizer.vocab["<eos>"]
    
    input_tensor = torch.tensor([[sos_idx]], dtype=torch.long).to(device)
    state = None
    generated_indices = []
    
    with torch.no_grad():
        for _ in range(20):
            logits, state = model_gen(input_tensor, state)
            logits = logits[0, -1, :]
            
            probs = torch.softmax(logits, dim=-1)
            next_idx = torch.multinomial(probs, num_samples=1).item()
            
            if next_idx == eos_idx:
                break
                
            generated_indices.append(next_idx)
            input_tensor = torch.tensor([[next_idx]], dtype=torch.long).to(device)
            
    return "".join(pokemon_tokenizer.decode(generated_indices)).capitalize()

In [ ]:
for _ in range(5):
    print(f"-> {generate_pokemon_name()}")

In [ ]:
pokemon_set = set(p.lower() for p in pokemons)

for _ in range(5):
    while True:
        name = generate_pokemon_name()
        
        # Só aceita se o nome gerado for completamente inédito
        if name.lower() not in pokemon_set:
            break
    print(f"-> {name}")

## Exercícios

### Exercício 1
Modifique o classificador de SMS para usar uma arquitetura de LSTM bidirecional (`bidirectional=True` ao instanciar `nn.LSTM`). Ajuste a camada linear `self.fc = nn.Linear(hidden_dim * 2, output_dim)`. Treine o modelo novamente e analise os efeitos na perda de validação.

### Exercício 2
Teste a geração autoregressiva com um dataset simples de frases, tokenizando por palavra ao invés de caractere.